In [1]:
import time
import pandas as pd
import io
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

print("Starting browser in headless mode...")
options = Options()
options.add_argument("--headless")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
dates = pd.date_range(start="2023-01-01", end="2023-12-31")
all_days = []

for i, date in enumerate(dates):
    date_str = date.strftime("%Y-%m-%d")
    url = f"https://www.wunderground.com/history/daily/gb/hounslow/EGLL/date/{date_str}"
    
    print(f"\n[{i+1}/365] Fetching data for {date_str}...")
    print(f" -> Loading URL: {url}")
    
    driver.get(url)
    
    # Wait for the JavaScript table to render
    print(" -> Waiting 5 seconds for JavaScript to render...")
    time.sleep(5) 
    
    try:
        print(" -> Parsing HTML for tables...")
        html = driver.page_source
        tables = pd.read_html(io.StringIO(html))
        
        print(f" -> Found {len(tables)} tables on the page.")
        
        # Search for the correct observation table dynamically
        observation_table = None
        for idx, table in enumerate(tables):
            # The correct table usually has a 'Time' column
            if 'Time' in table.columns or 'TIME' in table.columns:
                observation_table = table
                print(f" -> Extracted observation table (Index {idx}).")
                break
                
        if observation_table is not None:
            observation_table["Date"] = date_str
            all_days.append(observation_table)
            print(f" -> SUCCESS: Saved data for {date_str}")
        else:
            print(f" -> ERROR: Could not find the observation table in the HTML.")
            
    except Exception as e:
        print(f" -> FAILED on {date_str}. Error: {e}")

print("\nClosing browser...")
driver.quit()

if all_days:
    print("Merging data into CSV...")
    year_data = pd.concat(all_days, ignore_index=True)
    year_data.to_csv("wunderground_2023_EGLL.csv", index=False)
    print("Export complete: wunderground_2023_EGLL.csv")
else:
    print("No data was extracted. Weather Underground may be blocking the browser.")

Starting browser in headless mode...

[1/365] Fetching data for 2023-01-01...
 -> Loading URL: https://www.wunderground.com/history/daily/gb/hounslow/EGLL/date/2023-01-01
 -> Waiting 5 seconds for JavaScript to render...
 -> Parsing HTML for tables...
 -> Found 1 tables on the page.
 -> Extracted observation table (Index 0).
 -> SUCCESS: Saved data for 2023-01-01

[2/365] Fetching data for 2023-01-02...
 -> Loading URL: https://www.wunderground.com/history/daily/gb/hounslow/EGLL/date/2023-01-02
 -> Waiting 5 seconds for JavaScript to render...
 -> Parsing HTML for tables...
 -> Found 1 tables on the page.
 -> Extracted observation table (Index 0).
 -> SUCCESS: Saved data for 2023-01-02

[3/365] Fetching data for 2023-01-03...
 -> Loading URL: https://www.wunderground.com/history/daily/gb/hounslow/EGLL/date/2023-01-03
 -> Waiting 5 seconds for JavaScript to render...
 -> Parsing HTML for tables...
 -> Found 1 tables on the page.
 -> Extracted observation table (Index 0).
 -> SUCCESS: Sa

InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	(No symbol) [0x7ff71cbde745]
	(No symbol) [0x7ff71c793d00]
	(No symbol) [0x7ff71c5c3d36]
	(No symbol) [0x7ff71c610563]
	(No symbol) [0x7ff71c6489e2]
	(No symbol) [0x7ff71c6427a2]
	(No symbol) [0x7ff71c641f23]
	(No symbol) [0x7ff71c58c8eb]
	(No symbol) [0x7ff71c58cf52]
	(No symbol) [0x7ff71c58d585]
	(No symbol) [0x7ff71cc817b1]
	(No symbol) [0x7ff71ccb87c7]
	(No symbol) [0x7ff71ccac85e]
	(No symbol) [0x7ff71c95884e]
	(No symbol) [0x7ff71c7a0415]
	(No symbol) [0x7ff71c58bd1e]
	(No symbol) [0x7ff71cd34cff]
	(No symbol) [0x7ffcb319cd87]
	(No symbol) [0x7ffcb4d2caec]


In [ ]:
import pandas as pd

# Load the raw scraped data
df = pd.read_csv("wunderground_2023_EGLL.csv")

# 1. Process Text Data (Condition)
# Find the most frequent condition for the day
primary_condition = df.groupby('Date')['Condition'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else 'Unknown'
).rename('Primary Condition')

# Extract specific events based on keywords
def get_events(conditions):
    text = " ".join(conditions.dropna().unique()).lower()
    events = []
    if any(k in text for k in ['rain', 'drizzle', 'shower']): events.append('Rain')
    if 'snow' in text: events.append('Snow')
    if 'fog' in text: events.append('Fog')
    if 'thunder' in text or 't-storm' in text: events.append('Thunderstorm')
    return '-'.join(events) if events else 'None'

daily_events = df.groupby('Date')['Condition'].agg(get_events).rename('Events')

# 2. Process Numeric Data
unit_mapping = {
    'Temperature': '°C',
    'Dew Point': '°C',
    'Humidity': '%',
    'Wind Speed': 'km/h',
    'Wind Gust': 'km/h',
    'Pressure': 'hPa'
}

for col in unit_mapping.keys():
    df[col] = df[col].astype(str).str.extract(r'([\d\.]+)')[0].astype(float)

agg_funcs = {
    'Temperature': ['max', 'mean', 'min'],
    'Dew Point': ['max', 'mean', 'min'],
    'Humidity': ['max', 'mean', 'min'],
    'Wind Speed': ['max', 'mean'],
    'Wind Gust': ['max'],
    'Pressure': ['max', 'mean', 'min']
}

daily_summary = df.groupby('Date').agg(agg_funcs)

# Flatten columns and add units
new_columns = []
for col, stat in daily_summary.columns:
    stat_name = stat.capitalize()
    unit = unit_mapping[col]
    new_columns.append(f"{col} ({unit}) {stat_name}")

daily_summary.columns = new_columns

# 3. Merge Numeric and Text Summaries
daily_summary = daily_summary.round(1)
final_df = pd.concat([daily_summary, daily_events], axis=1).reset_index()

# Export
output_filename = "wunderground_2023_EGLL_Daily_Summary_with_Events.csv"
final_df.to_csv(output_filename, index=False)

print(f"Aggregation complete. Saved to {output_filename}")

Aggregation complete. Saved to wunderground_2023_EGLL_Daily_Summary_with_Events.csv


In [ ]:
df = pd.read_csv('wunderground_2023_EGLL_Daily_Summary_with_Events.csv')
print(df.shape)
df.head()

(365, 17)


,Date,Temperature (°C) Max,Temperature (°C) Mean,Temperature (°C) Min,Dew Point (°C) Max,Dew Point (°C) Mean,Dew Point (°C) Min,Humidity (%) Max,Humidity (%) Mean,Humidity (%) Min,Wind Speed (km/h) Max,Wind Speed (km/h) Mean,Wind Gust (km/h) Max,Pressure (hPa) Max,Pressure (hPa) Mean,Pressure (hPa) Min,Events
0,2023-01-01,12.0,10.4,7.0,9.0,7.1,6.0,93.0,80.1,71.0,31.0,18.5,46.0,1007.0,1004.8,1000.0,Rain
1,2023-01-02,8.0,5.8,2.0,6.0,3.7,1.0,100.0,87.0,70.0,19.0,10.4,NaN,1020.0,1013.9,1006.0,NaN
2,2023-01-03,13.0,8.8,3.0,11.0,7.5,2.0,100.0,92.2,87.0,33.0,20.1,52.0,1019.0,1014.9,1010.0,Rain
3,2023-01-04,14.0,12.5,11.0,11.0,9.1,7.0,94.0,80.1,67.0,44.0,30.2,NaN,1016.0,1011.4,1009.0,Rain
4,2023-01-05,14.0,11.5,10.0,10.0,8.4,7.0,94.0,81.5,72.0,28.0,20.7,41.0,1017.0,1015.0,1010.0,Rain
